# Network Capacity Forecasting — End-to-End Walkthrough

This notebook demonstrates the full pipeline:

1. Generate synthetic broadband telemetry
2. Engineer features in SQL
3. Fit and compare three forecasters
4. Detect anomalies on the residuals
5. Visualise results for stakeholder review

Designed to mirror what an analytics-track Network Data Scientist would do during a sprint.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from network_forecast import (
    GeneratorConfig,
    LightGBMForecaster,
    ResidualAnomalyDetector,
    SeasonalNaiveForecaster,
    generate_dataset,
    walk_forward_eval,
)
from network_forecast.sql_features import build_busy_hour_summary, build_daily_aggregates

## 1. Generate synthetic telemetry

In [ ]:
traffic, anomalies = generate_dataset(GeneratorConfig(random_seed=42))
print(f'{len(traffic):,} rows across {traffic["pop"].nunique()} PoPs')
print(f'{len(anomalies)} injected anomalies')
traffic.head()

## 2. SQL feature engineering

In [ ]:
traffic.to_parquet('../data/network_traffic.parquet', index=False)
busy = build_busy_hour_summary('../data/network_traffic.parquet')
busy.head(10)

## 3. Champion-challenger forecasting

In [ ]:
series = traffic[traffic['pop'] == 'LON-CORE-01'].set_index('timestamp')['traffic_gbps'].sort_index()

naive_results = walk_forward_eval(series, SeasonalNaiveForecaster, horizon=24, n_folds=4)
lgb_results = walk_forward_eval(series, lambda: LightGBMForecaster(num_boost_round=150), horizon=24, n_folds=4)

summary = pd.concat([naive_results, lgb_results]).groupby('forecaster_name')[['mae', 'rmse', 'smape']].mean().round(2)
summary

## 4. Anomaly detection

In [ ]:
train = series.iloc[:-24*14]
test = series.iloc[-24*14:]
model = SeasonalNaiveForecaster(season_length=168).fit(train)
expected = pd.Series(model.predict(len(test)).point_forecast, index=test.index)

train_residuals = (train.iloc[168:] - train.shift(168).iloc[168:]).values
detector = ResidualAnomalyDetector(threshold=3.5).fit(train_residuals)
detection = detector.detect(test.index, test.values, expected.values)
detection.to_dataframe().query('is_anomaly').head()